# 02_pipeline — governed DataFrame-to-table orchestration template

Read source data, register DataFrames, profile catalogue evidence, transform into target DataFrames, optionally curate governance through widgets, enforce guardrails, then write target data and runtime metadata.

Only edit the DataFrame reads, transformations, target write settings, and lineage relationships. Schema/freshness/profile/DQ guardrail authoring is handled by widgets after catalogue evidence exists.


## 1. Run `00_env_config`


In [ ]:
%run 00_env_config


## 2. Import required functions


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit.config import _current_audit_timestamp
from fabricops_kit.pipeline import display_guardrail_results

from fabricops_kit import (
    get_selected_agreement,
    prepare_pipeline_table_configs,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    run_table_guardrails,
    stop_if_failed,
    widget_author_dq_rules,
    widget_author_schema_freshness_profile_rules,
    widget_enrich_table_metadata,
    widget_review_guardrail_governance,
    widget_select_agreement,
    widget_select_guardrail_target,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)


## 3. Select agreement and capture run context


In [ ]:
PIPELINE_STARTED_AT = _current_audit_timestamp(config=CONFIG)
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = RUN_CONTEXT.runtime_metadata.get("currentNotebookName", "02_pipeline")
GUARDRAIL_DISPLAY_MODE = "summary"

widget_select_agreement(
    CONFIG,
    env_name=ENV_NAME,
    spark_session=spark,
    metadata_schema=METADATA_SCHEMA,
    register_notebook=True,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


---
# SOURCE AREA

Read source DataFrames, register key + DataFrame only, then profile catalogue evidence.


## 4. USER EDIT SECTION — read source DataFrames

The read call carries the physical source identity. Registration only needs a stable key and the DataFrame object.


In [ ]:
df_orders = read_lakehouse_table(
    CONFIG, ENV_NAME, "source", "demo_src_orders_happy", schema="DemoTest", spark_session=spark
)

df_customers = read_lakehouse_table(
    CONFIG, ENV_NAME, "source", "demo_src_customers_happy", schema="DemoTest", spark_session=spark
)

# Other examples:
# df_orders = read_lakehouse_csv(CONFIG, ENV_NAME, "source", "path/to/orders.csv", spark_session=spark, header=True)
# df_orders = read_lakehouse_parquet(CONFIG, ENV_NAME, "source", "path/to/orders.parquet", spark_session=spark)
# df_customers = read_lakehouse_excel(CONFIG, ENV_NAME, "source", "path/to/customers.xlsx", sheet_name=0, spark_session=spark)
# df_orders = read_warehouse_table(CONFIG, ENV_NAME, "source", "dbo", "orders", spark_session=spark)


## 5. USER EDIT SECTION — register source DataFrames only

Do not define schema, freshness, profile behaviour, DQ, classification, enrichment, or distribution settings here. Those are profiled and curated through widgets.


In [ ]:
SOURCE_TABLES = [
    {"key": "orders", "df": df_orders},
    {"key": "customers", "df": df_customers},
]

SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    SOURCE_TABLES,
    {},
    table_role="source",
)

df_orders = SOURCE_CONFIG_BY_KEY["orders"]["df"]
df_customers = SOURCE_CONFIG_BY_KEY["customers"]["df"]


## 6. Profile source DataFrames and write catalogue evidence

This records source-side evidence in `METADATA_DATA_CATALOGUE`. It is evidence generation, not manual schema authoring.


In [ ]:
source_profile_results = run_table_guardrails(
    SOURCE_TABLES, CONFIG, ENV_NAME,
    run_id=RUN_ID, pipeline_name=PIPELINE_NAME, notebook_id=NOTEBOOK_ID,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID, agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION, metadata_schema=METADATA_SCHEMA,
    table_role="source",
)

display_guardrail_results(source_profile_results, mode=GUARDRAIL_DISPLAY_MODE)


---
# TRANSFORMATION AND TARGET AREA


## 7. USER EDIT SECTION — transform source DataFrames into target DataFrames


In [ ]:
df_orders_enriched = (
    df_orders.alias("o")
    .join(df_customers.alias("c"), on="customer_id", how="left")
    .select(
        "order_id", "customer_id", "customer_name", "customer_segment",
        F.col("country_code").alias("order_country_code"),
        F.col("customer_country_code"), "order_date", "status", "order_amount",
        F.current_timestamp().alias("processed_ts"),
    )
)

df_orders_summary = (
    df_orders_enriched
    .groupBy("order_date", "customer_segment", "status")
    .agg(F.count("order_id").alias("order_count"), F.sum("order_amount").alias("total_order_amount"))
)


## 8. USER EDIT SECTION — register target DataFrames only

Targets can be profiled before physical target tables exist because the DataFrame already exists.


In [ ]:
TARGET_TABLES = [
    {"key": "orders_enriched", "df": df_orders_enriched},
    {"key": "orders_summary", "df": df_orders_summary},
]

TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    TARGET_TABLES,
    {},
    table_role="target",
    run_id=RUN_ID,
    pipeline_name=PIPELINE_NAME,
)

df_orders_enriched = TARGET_CONFIG_BY_KEY["orders_enriched"]["df"]
df_orders_summary = TARGET_CONFIG_BY_KEY["orders_summary"]["df"]


## 9. Profile target DataFrames and write catalogue evidence


In [ ]:
target_profile_results = run_table_guardrails(
    TARGET_TABLES, CONFIG, ENV_NAME,
    run_id=RUN_ID, pipeline_name=PIPELINE_NAME, notebook_id=NOTEBOOK_ID,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID, agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION, metadata_schema=METADATA_SCHEMA,
    table_role="target",
)

display_guardrail_results(target_profile_results, mode=GUARDRAIL_DISPLAY_MODE)


## 10. Optional governance curation and metadata enrichment widgets

Run only when you need to author or review schema/freshness/profile/DQ rules or enrich metadata from the latest catalogue evidence.


In [ ]:
selected_guardrail_target = widget_select_guardrail_target(
    CONFIG, env_name=ENV_NAME, spark_session=spark, metadata_schema=METADATA_SCHEMA
)

widget_author_schema_freshness_profile_rules(
    CONFIG, env_name=ENV_NAME, spark_session=spark, metadata_schema=METADATA_SCHEMA,
    selected_target=selected_guardrail_target,
)
widget_author_dq_rules(
    CONFIG, env_name=ENV_NAME, spark_session=spark, metadata_schema=METADATA_SCHEMA,
    selected_target=selected_guardrail_target,
)
widget_enrich_table_metadata(
    CONFIG, env_name=ENV_NAME, spark_session=spark, metadata_schema=METADATA_SCHEMA,
    selected_target=selected_guardrail_target,
)
widget_review_guardrail_governance(
    CONFIG, env_name=ENV_NAME, spark_session=spark, metadata_schema=METADATA_SCHEMA,
    selected_target=selected_guardrail_target,
)


## 11. Guardrail enforcement gate

If any blocking source or target guardrail fails, the notebook stops here before target write settings and before any target table write.


In [ ]:
source_enforcement_results = run_table_guardrails(
    SOURCE_TABLES, CONFIG, ENV_NAME,
    run_id=RUN_ID, pipeline_name=PIPELINE_NAME, notebook_id=NOTEBOOK_ID,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID, agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION, metadata_schema=METADATA_SCHEMA,
    table_role="source",
)
display_guardrail_results(source_enforcement_results, mode=GUARDRAIL_DISPLAY_MODE)
if not source_enforcement_results["can_continue"]:
    stop_if_failed({"status": "failed", "can_continue": False, "message": source_enforcement_results.get("blocking_message") or "Blocking source guardrail failure."})

target_enforcement_results = run_table_guardrails(
    TARGET_TABLES, CONFIG, ENV_NAME,
    run_id=RUN_ID, pipeline_name=PIPELINE_NAME, notebook_id=NOTEBOOK_ID,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID, agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION, metadata_schema=METADATA_SCHEMA,
    table_role="target",
)
display_guardrail_results(target_enforcement_results, mode=GUARDRAIL_DISPLAY_MODE)
if not target_enforcement_results["can_continue"]:
    stop_if_failed({"status": "failed", "can_continue": False, "message": target_enforcement_results.get("blocking_message") or "Blocking target guardrail failure."})


## 12. USER EDIT SECTION — configure target write settings

Write settings belong after the guardrail gate. `target_name` and `write_mode` are essential. Layer, schema, options, partitioning, and repartitioning are write concerns only.


In [ ]:
TARGET_WRITE_SETTINGS = {
    "orders_enriched": {
        "target_layer": "unified",
        "target_name": "demo_unified_orders_enriched",
        "schema": "DemoTest",
        "write_mode": "overwrite",
        "options": {"overwriteSchema": "true"},
    },
    "orders_summary": {
        "target_layer": "unified",
        "target_name": "demo_unified_orders_summary",
        "schema": "DemoTest",
        "write_mode": "overwrite",
        "options": {"overwriteSchema": "true"},
        # Optional: "partition_by": ["order_date"],
        # Optional: "repartition_by": ["customer_segment"],
    },
}

for key, write_settings in TARGET_WRITE_SETTINGS.items():
    TARGET_CONFIG_BY_KEY[key].update(write_settings)


## 13. Write target Lakehouse tables

This section only runs after all blocking guardrails pass.


In [ ]:
target_write_status = {}

for key, target in TARGET_CONFIG_BY_KEY.items():
    write_lakehouse_table(
        target["df"],
        CONFIG,
        ENV_NAME,
        target.get("target_layer", "unified"),
        target["target_name"],
        schema=target.get("schema"),
        mode=target.get("write_mode", "overwrite"),
        partition_by=target.get("partition_by"),
        repartition_by=target.get("repartition_by"),
        options=target.get("options"),
    )
    target_write_status[key] = f"written: {target.get('schema')}.{target['target_name']}"

target_write_status


## 14. Optional warehouse write example


In [ ]:
# orders_summary_target = TARGET_CONFIG_BY_KEY["orders_summary"]
# write_warehouse_table(
#     orders_summary_target["df"], CONFIG, ENV_NAME, "product", "dbo",
#     orders_summary_target["target_name"],
#     mode=orders_summary_target.get("write_mode", "overwrite"),
# )


## 15. USER EDIT SECTION — lineage relationships


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {"source_key": "orders", "target_key": "orders_enriched", "transformation_type": "join", "transformation_logic": "orders joined to customers on customer_id"},
    {"source_key": "customers", "target_key": "orders_enriched", "transformation_type": "join", "transformation_logic": "customers joined to orders on customer_id"},
    {"source_key": "orders_enriched", "target_key": "orders_summary", "transformation_type": "aggregation", "transformation_logic": "group by order_date, customer_segment, and status"},
]


## 16. Write lineage metadata


In [ ]:
lineage_result = write_pipeline_lineage(
    LINEAGE_RELATIONSHIPS, SOURCE_CONFIG_BY_KEY, TARGET_CONFIG_BY_KEY, CONFIG, ENV_NAME,
    run_id=RUN_ID, pipeline_name=PIPELINE_NAME, notebook_id=NOTEBOOK_ID,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID, agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION, metadata_schema=METADATA_SCHEMA,
)
lineage_result


## 17. Write runtime summary


In [ ]:
pipeline_status = "succeeded" if all(
    result.get("can_continue", False) for result in [source_enforcement_results, target_enforcement_results]
) else "failed"

runtime_summary_result = write_pipeline_run_summary(
    CONFIG, ENV_NAME, run_id=RUN_ID, pipeline_name=PIPELINE_NAME,
    pipeline_started_at=PIPELINE_STARTED_AT, completed_at=_current_audit_timestamp(config=CONFIG), status=pipeline_status,
    source_guardrail_results=source_enforcement_results, target_guardrail_results=target_enforcement_results,
    source_tables=SOURCE_CONFIG_BY_KEY, target_tables=TARGET_CONFIG_BY_KEY,
    target_write_status=target_write_status, lineage_result=lineage_result,
    notebook_id=NOTEBOOK_ID, notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    agreement_id=AGREEMENT_ID, agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    metadata_schema=METADATA_SCHEMA,
)
runtime_summary_result
